In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

# M1 GPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [2]:
train_df = pd.read_csv("../data/processed/train.csv")
val_df   = pd.read_csv("../data/processed/val.csv")
test_df  = pd.read_csv("../data/processed/test.csv")

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 34217 | Val: 4277 | Test: 4278


In [3]:
MODEL_NAME = "bert-base-uncased"
tokenizer  = BertTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts, max_len=128):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

# Quick test
sample = tokenize(["You won't believe what happened!"])
print("Tokenizer OK — input_ids shape:", sample["input_ids"].shape)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer OK — input_ids shape: torch.Size([1, 10])


In [4]:
class ClickbaitDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.encodings = tokenizer(
            list(df["headline"]),
            padding=True,
            truncation=True,
            max_length=max_len
        )
        self.labels = list(df["label"])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = ClickbaitDataset(train_df, tokenizer)
val_dataset   = ClickbaitDataset(val_df,   tokenizer)
test_dataset  = ClickbaitDataset(test_df,  tokenizer)

print(f"Datasets ready — Train: {len(train_dataset)} | Val: {len(val_dataset)}")

Datasets ready — Train: 34217 | Val: 4277


In [5]:
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)
model = model.to(device)
print("Model loaded on", device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on mps


In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds, average="weighted")
    }

In [9]:
training_args = TrainingArguments(
    output_dir                  = "../models/bert_classifier",
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    warmup_steps                = 100,
    weight_decay                = 0.01,
    logging_dir                 = "../models/bert_classifier/logs",
    logging_steps               = 50,
    eval_strategy               = "epoch",   # renamed from evaluation_strategy
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
)

print("Training args OK")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training args OK


In [10]:
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics
)

trainer.train()

/Users/viki/Desktop/clickbait-detector/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.242846,0.240436,0.902268,0.902718
2,0.185364,0.293686,0.917232,0.916435
3,0.087831,0.423429,0.908347,0.908301


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/viki/Desktop/clickbait-detector/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/viki/Desktop/clickbait-detector/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=6417, training_loss=0.18237738437847276, metrics={'train_runtime': 3621.2722, 'train_samples_per_second': 28.347, 'train_steps_per_second': 1.772, 'total_flos': 6752153235939840.0, 'train_loss': 0.18237738437847276, 'epoch': 3.0})

In [11]:
preds_output = trainer.predict(test_dataset)
preds        = np.argmax(preds_output.predictions, axis=1)
labels       = preds_output.label_ids

print(classification_report(labels, preds, target_names=["Not Clickbait", "Clickbait"]))

/Users/viki/Desktop/clickbait-detector/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


               precision    recall  f1-score   support

Not Clickbait       0.91      0.96      0.94      2676
    Clickbait       0.93      0.85      0.89      1602

     accuracy                           0.92      4278
    macro avg       0.92      0.90      0.91      4278
 weighted avg       0.92      0.92      0.92      4278



In [12]:
model.save_pretrained("../models/bert_classifier")
tokenizer.save_pretrained("../models/bert_classifier")
print("Model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!


In [13]:
def predict(headline):
    inputs = tokenizer(
        headline,
        return_tensors  = "pt",
        truncation      = True,
        max_length      = 128,
        padding         = True
    ).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    probs     = torch.softmax(outputs.logits, dim=1)
    label_idx = torch.argmax(probs).item()
    confidence = probs[0][label_idx].item()

    label = "CLICKBAIT" if label_idx == 1 else "NOT CLICKBAIT"
    print(f"Headline  : {headline}")
    print(f"Prediction: {label} ({confidence*100:.1f}% confidence)")

# Test it
predict("You won't believe what this student built with AI!")
predict("Researchers publish new study on climate change impact.")

Headline  : You won't believe what this student built with AI!
Prediction: NOT CLICKBAIT (50.5% confidence)
Headline  : Researchers publish new study on climate change impact.
Prediction: NOT CLICKBAIT (99.8% confidence)
